# 02 · Partitions, keys and ordering
**Level:** beginner · **Time:** 40 minutes · **Prerequisite:** 01.
You will inspect partition metadata and test per-key order. You will also see why
adding partitions can change where future keyed events go.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


In [ ]:
require_broker()
topic = new_topic('partitioning', partitions=3)
print(topic)


## 1. Inspect the partition leaders
The one-broker setup has one replica per partition. Three partitions on one broker
increase logical parallelism; they do not provide broker-failure redundancy.


In [ ]:
from confluent_kafka.admin import AdminClient
admin = AdminClient(client_config())
info = admin.list_topics(topic=topic, timeout=10).topics[topic]
for number, part in info.partitions.items():
    print({"partition": number, "leader": part.leader, "replicas": part.replicas, "isr": part.isrs})
assert len(info.partitions) == 3


## 2. Key by the entity whose order matters
A stable nonempty key uses the client's partitioner to select a partition. Different keys
may collide on the same partition. This exercise does not assume a particular hash algorithm.


In [ ]:
events = [{"event_id": str(uuid4()), "customer_id": customer, "sequence": sequence}
          for sequence in range(4) for customer in ["alice", "bob", "carol", "dev"]]
publish(topic, events, key_field="customer_id")
messages = collect(topic, count=len(events))
[(m.key().decode(), m.partition(), m.offset(), decode(m)["sequence"]) for m in messages]


## 3. Test the invariant, not a lucky output order
The consumer can interleave records from different partitions. There is no single
global offset. For each customer, a single producer sends sequence 0, 1, 2, 3 in order.


In [ ]:
from collections import defaultdict
by_customer = defaultdict(list)
for m in messages:
    by_customer[m.key().decode()].append((m.partition(), decode(m)["sequence"]))
for customer, rows in by_customer.items():
    assert len({partition for partition, _ in rows}) == 1
    assert [sequence for _, sequence in rows] == [0, 1, 2, 3]
    print(customer, rows)


## 4. Explicit routing
Use explicit partitions to demonstrate their independent offsets. Most business producers
should rely on a deliberate key strategy instead of hard-coding partition numbers.


In [ ]:
import json
explicit = new_topic("explicit-routing", partitions=3)
p = producer()
errors = []
def report(error, message):
    if error:
        errors.append(str(error))
for partition in range(3):
    for sequence in range(2):
        p.produce(explicit, partition=partition,
                  value=json.dumps({"partition_requested": partition, "sequence": sequence}).encode(),
                  on_delivery=report)
assert p.flush(15) == 0 and not errors, errors
rows = collect(explicit, count=6)
for partition in range(3):
    offsets = [m.offset() for m in rows if m.partition() == partition]
    assert offsets == [0, 1]
    print("Partition", partition, "has offsets", offsets)


## 5. Observe a different partition count
Create a second topic with six partitions. Which keys move depends on hashing; no
particular key is guaranteed to move. Existing records never move merely because partitions are added.


In [ ]:
larger = new_topic("partitioning-six", partitions=6)
publish(larger, events, key_field="customer_id")
larger_rows = collect(larger, count=len(events))
old_map = {m.key().decode(): m.partition() for m in messages}
new_map = {m.key().decode(): m.partition() for m in larger_rows}
for key in sorted(old_map):
    print(key, "3 partitions:", old_map[key], "6 partitions:", new_map[key])


## Expected observation
Each nonempty key stays on one partition for a fixed topic partition count and partitioner.
Every partition has its own offset sequence. Multiple producers or a partition-count
change require additional business sequencing if strict entity order is essential.

## Exercise
Choose a key for (a) a bank account ledger, (b) shipment tracking, and (c) click analytics.
Then modify the dataset so 95% of records use one customer key. Explain the bottleneck.
<details><summary>Solution hints</summary>
Account ID, shipment ID, and a carefully chosen user/session key are possible choices.
One hot key stays on one partition, so simply adding group members does not split its load.
Salting a key spreads load but gives up straightforward per-entity ordering.
</details>
Next: **03 — delivery acknowledgements and producer failures**.
